# NewsAPI + Market Data Alignment

Use this notebook to fetch NewsAPI articles, cache them locally, align them to your historical market candles, and compare news intensity/sentiment/language vectors with price returns.

The notebook does not store your API key. Set `NEWSAPI_KEY` in your shell before launching Jupyter, or paste it into the config cell temporarily.

In [ ]:
import sys, os, json, math, time, re, hashlib
from pathlib import Path
from datetime import datetime, timezone, timedelta

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import requests
import torch

from rl_trading_playground.data import PAIRS

from bokeh.palettes import Category10
import bokeh.plotting as bk
from bokeh.layouts import column
bk.output_notebook()

print("project root:", PROJECT_ROOT)

## Configuration

`/v2/everything` is the useful NewsAPI endpoint here because it supports keyword searches and date ranges. Keep the first experiments small: one or two assets, a short date range, and hourly aggregation.

In [ ]:
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", "dc7e58fe92cd408eafbb736407c04a29")  # or paste temporarily: "your_key_here"

DATA_PATH = PROJECT_ROOT / "data" / "historical_data0.ptt"
NEWS_CACHE_DIR = PROJECT_ROOT / "data" / "newsapi"
NEWS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Choose assets after loading the market cache. These names match the current pairs dict if present.
SELECTED_ASSETS = ["Bitcoin", "Ethereum", "Solana"]

# Keep this narrow at first; NewsAPI plans may limit historical depth and request volume.
# If these dates do not overlap your market cache, a later cell moves them to the market-data tail.
NEWS_FROM = "2025-03-15"
NEWS_TO = "2025-03-31"
LANGUAGE = "en"
PAGE_SIZE = 100
MAX_PAGES_PER_QUERY = 2
AGG_FREQ = "1h"  # examples: "15min", "1h", "4h", "1D"
USE_MACRO_FILTER = False

ASSET_QUERIES = {
    "Bitcoin": '(bitcoin OR BTC OR "Bitcoin ETF" OR "crypto regulation")',
    "Ethereum": '(ethereum OR ether OR ETH OR "Ethereum ETF" OR "smart contracts")',
    "Solana": '(solana OR SOL OR "Solana ETF")',
    "Cardano": '(cardano OR ADA)',
    "Ripple": '(ripple OR XRP OR "Ripple Labs")',
    "EURUSD": '("European Central Bank" OR ECB OR Lagarde OR eurozone OR EUR OR "Federal Reserve" OR Powell OR USD)',
    "GBPUSD": '("Bank of England" OR BoE OR sterling OR GBP OR "Federal Reserve" OR Powell OR USD)',
    "USDJPY": '("Bank of Japan" OR BoJ OR yen OR JPY OR "Federal Reserve" OR Powell OR USD)',
}

MACRO_QUERY = '(inflation OR "interest rates" OR "central bank" OR recession OR "risk appetite" OR "geopolitical risk")'

print("NewsAPI key set:", bool(NEWSAPI_KEY))
print("Cache dir:", NEWS_CACHE_DIR.resolve())

## Load Market Data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run your data preparation first.")

raw = torch.load(DATA_PATH, map_location="cpu")
times = pd.to_datetime(raw["times"].cpu().numpy(), unit="s", utc=True)

close = raw["close"].cpu().numpy()
open_ = raw.get("open", raw["close"]).cpu().numpy()
high = raw.get("high", raw["close"]).cpu().numpy()
low = raw.get("low", raw["close"]).cpu().numpy()
volume = raw["volume"].cpu().numpy()

pairs = raw.get("pairs", {})
print("close shape:", close.shape)
print("stored pairs:", pairs)

if isinstance(pairs, dict) and len(pairs) == close.shape[1]:
    asset_names = list(pairs.keys())
    symbols = list(pairs.values())
elif len(PAIRS) >= close.shape[1]:
    asset_names = list(PAIRS.keys())[: close.shape[1]]
    symbols = list(PAIRS.values())[: close.shape[1]]
    print("Stored pair metadata length does not match data width; using rl_trading_playground.data.PAIRS order.")
else:
    asset_names = [f"asset_{i}" for i in range(close.shape[1])]
    symbols = asset_names
    print("Pair metadata length does not match data width; using generic asset names.")

market = pd.DataFrame(index=times)
for i, name in enumerate(asset_names):
    market[(name, "open")] = open_[:, i]
    market[(name, "high")] = high[:, i]
    market[(name, "low")] = low[:, i]
    market[(name, "close")] = close[:, i]
    market[(name, "volume")] = volume[:, i]
market.columns = pd.MultiIndex.from_tuples(market.columns, names=["asset", "field"])
market = market.sort_index()

available_assets = list(market.columns.get_level_values("asset").unique())
SELECTED_ASSETS = [a for a in SELECTED_ASSETS if a in available_assets]
if not SELECTED_ASSETS:
    SELECTED_ASSETS = available_assets[: min(3, len(available_assets))]

market_start = market.index.min()
market_end = market.index.max()
news_start = pd.Timestamp(NEWS_FROM, tz="UTC")
news_end = pd.Timestamp(NEWS_TO, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
if news_end < market_start or news_start > market_end:
    adjusted_end = market_end.floor("D")
    adjusted_start = max(market_start.floor("D"), adjusted_end - pd.Timedelta(days=14))
    NEWS_FROM = adjusted_start.strftime("%Y-%m-%d")
    NEWS_TO = adjusted_end.strftime("%Y-%m-%d")
    print("Adjusted NEWS_FROM/NEWS_TO to overlap market data:", NEWS_FROM, "->", NEWS_TO)

print("available assets:", available_assets)
print("selected assets:", SELECTED_ASSETS)
print("market date range:", market_start, "->", market_end)
print("news query range:", NEWS_FROM, "->", NEWS_TO)

## NewsAPI Fetch + Cache

The cache key includes query and date parameters. Re-running the notebook reuses saved responses unless `force_refresh=True`.

In [ ]:
def cache_path_for(query, from_date, to_date, language):
    key = json.dumps({"q": query, "from": from_date, "to": to_date, "language": language}, sort_keys=True)
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]
    return NEWS_CACHE_DIR / f"newsapi_{digest}.json"


def fetch_newsapi_everything(query, from_date, to_date, language="en", page_size=100, max_pages=1, force_refresh=False):
    cache_path = cache_path_for(query, from_date, to_date, language)
    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text())

    if not NEWSAPI_KEY:
        raise ValueError("Set NEWSAPI_KEY in the config cell or as an environment variable before fetching.")

    all_articles = []
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "from": from_date,
            "to": to_date,
            "language": language,
            "sortBy": "publishedAt",
            "pageSize": page_size,
            "page": page,
            "apiKey": NEWSAPI_KEY,
        }
        r = requests.get("https://newsapi.org/v2/everything", params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"NewsAPI error {r.status_code}: {r.text[:500]}")
        payload = r.json()
        articles = payload.get("articles", [])
        all_articles.extend(articles)
        if len(articles) < page_size:
            break
        time.sleep(0.25)

    result = {"query": query, "from": from_date, "to": to_date, "language": language, "articles": all_articles}
    cache_path.write_text(json.dumps(result, indent=2))
    return result


def normalize_articles(payload, asset):
    rows = []
    for a in payload.get("articles", []):
        source = a.get("source") or {}
        rows.append({
            "asset": asset,
            "source": source.get("name"),
            "author": a.get("author"),
            "title": a.get("title") or "",
            "description": a.get("description") or "",
            "content": a.get("content") or "",
            "url": a.get("url"),
            "published_at": pd.to_datetime(a.get("publishedAt"), utc=True, errors="coerce"),
        })
    return rows


def dedupe_articles(df):
    if df.empty:
        return df
    key = df["url"].fillna("") + "|" + df["title"].fillna("")
    return df.loc[~key.duplicated()].copy()

In [ ]:
article_rows = []
queries_used = {}

for asset in SELECTED_ASSETS:
    asset_query = ASSET_QUERIES.get(asset, asset)
    query = f"({asset_query}) AND ({MACRO_QUERY})" if USE_MACRO_FILTER else asset_query
    queries_used[asset] = query
    print("Fetching", asset, "->", query)
    payload = fetch_newsapi_everything(
        query,
        NEWS_FROM,
        NEWS_TO,
        language=LANGUAGE,
        page_size=PAGE_SIZE,
        max_pages=MAX_PAGES_PER_QUERY,
        force_refresh=False,
    )
    article_rows.extend(normalize_articles(payload, asset))

articles = dedupe_articles(pd.DataFrame(article_rows))
if not articles.empty:
    articles = articles.dropna(subset=["published_at"]).sort_values("published_at")
    articles["text"] = (articles["title"].fillna("") + ". " + articles["description"].fillna("") + ". " + articles["content"].fillna(""))

print("articles:", len(articles))
display(articles.head(10) if not articles.empty else articles)

## Lightweight Text Features

This uses a small financial sentiment lexicon and a hashing vectorizer implemented with standard Python. It is deliberately simple and reproducible. Later, you can replace `hashing_embedding` with FinBERT, OpenAI embeddings, sentence-transformers, or your own encoder.

In [ ]:
POSITIVE_WORDS = {
    "beat", "beats", "bullish", "gain", "gains", "growth", "higher", "increase", "increases",
    "optimism", "outperform", "rally", "record", "recover", "recovery", "strong", "surge", "upbeat",
    "approval", "approved", "cut", "cuts", "easing", "dovish", "cooling", "stabilize", "stabilized",
}
NEGATIVE_WORDS = {
    "bearish", "crash", "crisis", "decline", "declines", "default", "drop", "drops", "fear", "fall",
    "falls", "fraud", "hack", "hacked", "higher inflation", "lawsuit", "loss", "losses", "miss", "misses",
    "plunge", "risk", "risks", "selloff", "slump", "weak", "weaker", "war", "hawkish", "tightening",
}
TOKEN_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9_\-]+")


def tokenize(text):
    return [t.lower() for t in TOKEN_RE.findall(str(text))]


def lexicon_sentiment(text):
    toks = tokenize(text)
    if not toks:
        return 0.0
    pos = sum(t in POSITIVE_WORDS for t in toks)
    neg = sum(t in NEGATIVE_WORDS for t in toks)
    return (pos - neg) / math.sqrt(len(toks))


def hashing_embedding(text, dim=128):
    vec = np.zeros(dim, dtype=np.float32)
    toks = tokenize(text)
    for tok in toks:
        h = int(hashlib.md5(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if ((h >> 8) & 1) else -1.0
        vec[idx] += sign
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

if articles.empty:
    print("No articles fetched. Adjust queries/date range or check your NewsAPI plan.")
else:
    articles["sentiment"] = articles["text"].map(lexicon_sentiment)
    articles["text_len"] = articles["text"].map(lambda s: len(tokenize(s)))
    embeddings = np.vstack([hashing_embedding(t, dim=128) for t in articles["text"]])
    print("embedding matrix:", embeddings.shape)
    display(articles[["published_at", "asset", "source", "title", "sentiment", "text_len"]].head(10))

## Aggregate News By Time Bucket

In [ ]:
def aggregate_news(articles, embeddings=None, freq="1h"):
    if articles.empty:
        return pd.DataFrame(), np.empty((0, 128), dtype=np.float32)

    df = articles.copy().reset_index(drop=True)
    df["bucket"] = df["published_at"].dt.floor(freq)
    grouped = df.groupby(["asset", "bucket"], observed=True)

    feats = grouped.agg(
        news_count=("title", "size"),
        sentiment_mean=("sentiment", "mean"),
        sentiment_sum=("sentiment", "sum"),
        avg_text_len=("text_len", "mean"),
    ).reset_index()

    if embeddings is None:
        return feats, None

    emb_rows = []
    emb_keys = []
    for key, idx in grouped.indices.items():
        emb_keys.append(key)
        emb_rows.append(embeddings[list(idx)].mean(axis=0))
    emb_df = pd.DataFrame(emb_keys, columns=["asset", "bucket"])
    emb = np.vstack(emb_rows).astype(np.float32) if emb_rows else np.empty((0, embeddings.shape[1]), dtype=np.float32)
    feats = feats.merge(emb_df.assign(_emb_row=np.arange(len(emb_df))), on=["asset", "bucket"], how="left")
    return feats, emb

news_features, news_bucket_embeddings = aggregate_news(articles, embeddings if not articles.empty else None, AGG_FREQ)
print(news_features.shape, news_bucket_embeddings.shape if news_bucket_embeddings is not None else None)
display(news_features.head(10))

## Aggregate Market Data To The Same Frequency

In [ ]:
def aggregate_market(market, assets, freq="1h"):
    rows = []
    for asset in assets:
        px = market[asset].copy()
        agg = pd.DataFrame({
            "open": px["open"].resample(freq).first(),
            "high": px["high"].resample(freq).max(),
            "low": px["low"].resample(freq).min(),
            "close": px["close"].resample(freq).last(),
            "volume": px["volume"].resample(freq).sum(),
        }).dropna(subset=["close"])
        agg["asset"] = asset
        agg["bucket"] = agg.index
        agg["return_1"] = agg["close"].pct_change()
        agg["return_fwd_1"] = agg["close"].pct_change().shift(-1)
        agg["return_fwd_4"] = agg["close"].pct_change(4).shift(-4)
        agg["volatility_24"] = agg["return_1"].rolling(24, min_periods=4).std()
        rows.append(agg.reset_index(drop=True))
    return pd.concat(rows, ignore_index=True)

market_features = aggregate_market(market, SELECTED_ASSETS, AGG_FREQ)
print(market_features.shape)
display(market_features.head())

## Join News And Market Features

In [ ]:
aligned = market_features.merge(news_features, on=["asset", "bucket"], how="left")
for col in ["news_count", "sentiment_mean", "sentiment_sum", "avg_text_len"]:
    aligned[col] = aligned[col].fillna(0.0)

# Rolling news memory: what the agent would know from recent prior buckets.
for asset, idx in aligned.groupby("asset").groups.items():
    idx = list(idx)
    aligned.loc[idx, "news_count_24"] = aligned.loc[idx, "news_count"].rolling(24, min_periods=1).sum().values
    aligned.loc[idx, "sentiment_24"] = aligned.loc[idx, "sentiment_sum"].rolling(24, min_periods=1).sum().values
    aligned.loc[idx, "sentiment_72"] = aligned.loc[idx, "sentiment_sum"].rolling(72, min_periods=1).sum().values

print(aligned.shape)
display(aligned.head(10))

## Visual Comparison

These plots overlay price, forward return, article count, and rolling sentiment for each selected asset.

In [ ]:
def plot_asset_alignment(aligned, asset):
    df = aligned[aligned["asset"] == asset].copy().sort_values("bucket")
    if df.empty:
        print("No data for", asset)
        return

    palette = Category10[10]
    p1 = bk.figure(title=f"{asset} close", x_axis_type="datetime", width=1100, height=260)
    p1.line(df["bucket"], df["close"], line_width=2, color=palette[0], legend_label="close")
    p1.legend.click_policy = "hide"

    p2 = bk.figure(title=f"{asset} returns and news", x_axis_type="datetime", width=1100, height=300, x_range=p1.x_range)
    p2.line(df["bucket"], df["return_fwd_1"].fillna(0), line_width=1.5, color=palette[1], legend_label="next bucket return")
    p2.vbar(df["bucket"], top=df["news_count"], width=60 * 60 * 1000 * 0.7, alpha=0.25, color=palette[2], legend_label="news count")
    p2.line(df["bucket"], df["sentiment_24"], line_width=2, color=palette[3], legend_label="24 bucket sentiment sum")
    p2.legend.click_policy = "hide"

    bk.show(column(p1, p2))

for asset in SELECTED_ASSETS:
    plot_asset_alignment(aligned, asset)

## Correlation Scan

This is not a trading signal by itself. It is a quick diagnostic for whether the news features have any visible relationship with forward returns at this aggregation.

In [ ]:
feature_cols = ["news_count", "news_count_24", "sentiment_sum", "sentiment_24", "sentiment_72"]
target_cols = ["return_fwd_1", "return_fwd_4", "volatility_24"]

corr_rows = []
for asset, df in aligned.groupby("asset"):
    for f in feature_cols:
        for y in target_cols:
            tmp = df[[f, y]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 20 or tmp[f].std() == 0 or tmp[y].std() == 0:
                corr = np.nan
            else:
                corr = tmp[f].corr(tmp[y])
            corr_rows.append({"asset": asset, "feature": f, "target": y, "corr": corr, "n": len(tmp)})

corr_df = pd.DataFrame(corr_rows).sort_values("corr", key=lambda s: s.abs(), ascending=False)
display(corr_df)

## Cross-Attention Prototype

This cell demonstrates the tensor shape you could later plug into a policy/value model. Market tokens are recent return/volume/volatility buckets. News tokens are article text embeddings from the previous lookback window. This is intentionally a prototype, not a trained forecasting model.

In [ ]:
def build_market_tokens(aligned, asset, end_time, lookback=64):
    df = aligned[(aligned["asset"] == asset) & (aligned["bucket"] <= end_time)].sort_values("bucket").tail(lookback)
    cols = ["return_1", "volume", "volatility_24", "news_count_24", "sentiment_24"]
    x = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
    if len(x) < lookback:
        pad = np.zeros((lookback - len(x), x.shape[1]), dtype=np.float32)
        x = np.vstack([pad, x])
    return torch.tensor(x)


def build_article_tokens(articles, article_embeddings, asset, end_time, hours=72, max_articles=32):
    if articles.empty:
        return torch.zeros((0, 128), dtype=torch.float32)
    start = end_time - pd.Timedelta(hours=hours)
    mask = (articles["asset"] == asset) & (articles["published_at"] > start) & (articles["published_at"] <= end_time)
    idx = articles.reset_index(drop=True).index[mask.reset_index(drop=True)].to_numpy()
    idx = idx[-max_articles:]
    if len(idx) == 0:
        return torch.zeros((0, article_embeddings.shape[1]), dtype=torch.float32)
    return torch.tensor(article_embeddings[idx], dtype=torch.float32)


def demo_cross_attention(market_tokens, news_tokens, d_model=64):
    # Random projections only demonstrate mechanics. In a model these are trainable layers.
    torch.manual_seed(7)
    Wq = torch.randn(market_tokens.shape[-1], d_model) / math.sqrt(market_tokens.shape[-1])
    Wk = torch.randn(news_tokens.shape[-1], d_model) / math.sqrt(news_tokens.shape[-1])
    Wv = torch.randn(news_tokens.shape[-1], d_model) / math.sqrt(news_tokens.shape[-1])
    Q = market_tokens @ Wq
    K = news_tokens @ Wk
    V = news_tokens @ Wv
    weights = torch.softmax((Q @ K.T) / math.sqrt(d_model), dim=-1)
    context = weights @ V
    return context, weights

asset = SELECTED_ASSETS[0]
end_time = aligned[aligned["asset"] == asset]["bucket"].max()
market_tokens = build_market_tokens(aligned, asset, end_time, lookback=64)
news_tokens = build_article_tokens(articles, embeddings if not articles.empty else np.empty((0,128), dtype=np.float32), asset, end_time, hours=72, max_articles=32)

print("asset:", asset)
print("market_tokens:", tuple(market_tokens.shape), "news_tokens:", tuple(news_tokens.shape))
if len(news_tokens) > 0:
    context, attn_weights = demo_cross_attention(market_tokens, news_tokens)
    print("context:", tuple(context.shape), "attention:", tuple(attn_weights.shape))
else:
    print("No recent news tokens for the selected end time. Try widening NEWS_FROM/NEWS_TO or changing the asset.")

## Export Aligned Features

This file can become an auxiliary data source for an RL training notebook. The raw article cache remains in `data/newsapi/`.

In [ ]:
EXPORT_PATH = NEWS_CACHE_DIR / f"aligned_news_market_{AGG_FREQ}.parquet"
try:
    aligned.to_parquet(EXPORT_PATH, index=False)
    print("wrote", EXPORT_PATH.resolve())
except Exception as exc:
    EXPORT_PATH = NEWS_CACHE_DIR / f"aligned_news_market_{AGG_FREQ}.csv"
    aligned.to_csv(EXPORT_PATH, index=False)
    print("Parquet unavailable; wrote CSV instead:", EXPORT_PATH.resolve())
    print(type(exc).__name__, exc)

## Next Experiments

1. Replace the hashing vectors with a stronger embedding model.
2. Use publication-time latency buffers, for example only articles older than 5-15 minutes at each decision time.
3. Train a market-only baseline and a market+news baseline on the same split.
4. Add asset relevance routing so unrelated macro/news text does not flood every asset.
5. Move from aggregate features to article-token cross-attention once the simple features show signal.